# DTW+hier, ligação média, dV, STL

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais para dV
agg_pivot_dv = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot_dv.index.tolist()
X = agg_pivot_dv.values.astype(float)
n = X.shape[0]


# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL das séries médias por cluster
# ==============================

# Ordenar clusters: cluster 1 vem sempre primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered  # cluster 1 na primeira coluna

fig_stl, axes = plt.subplots(3, n_clusters, figsize=(5 * n_clusters, 10), sharex=True)

# Garantir formato correto se houver apenas um cluster
if n_clusters == 1:
    axes = np.array([axes]).T

# Calcular decomposições STL e limites globais
stl_results = {}
trend_min, trend_max = np.inf, -np.inf
season_min, season_max = np.inf, -np.inf
resid_min, resid_max = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)

    ts = pd.Series(cluster_mean_dV.values, index=pd.to_datetime(cluster_mean_dV.index)).asfreq('MS').interpolate()
    stl = STL(ts, period=12, robust=True)
    res = stl.fit()
    stl_results[cluster_id] = res

    # Atualizar limites globais para manter mesma escala entre clusters
    trend_min = min(trend_min, res.trend.min())
    trend_max = max(trend_max, res.trend.max())
    season_min = min(season_min, res.seasonal.min())
    season_max = max(season_max, res.seasonal.max())
    resid_min = min(resid_min, res.resid.min())
    resid_max = max(resid_max, res.resid.max())

# Plot de cada cluster (mesma cor por cluster)
for col_idx, cluster_id in enumerate(clusters_ordered):
    res = stl_results[cluster_id]
    cluster_color = cluster_colors[cluster_id]

    # Tendência
    ax_trend = axes[0, col_idx]
    ax_trend.plot(res.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_title(f"Tendência – Cluster {cluster_id}", fontsize=12)
    ax_trend.set_ylim(trend_min, trend_max)
    if col_idx == 0:
        ax_trend.set_ylabel("dV (mm)")

    # Sazonal
    ax_seasonal = axes[1, col_idx]
    ax_seasonal.plot(res.seasonal, color=cluster_color, linewidth=2.0)
    ax_seasonal.set_title(f"Sazonal – Cluster {cluster_id}", fontsize=12)
    ax_seasonal.set_ylim(season_min, season_max)
    if col_idx == 0:
        ax_seasonal.set_ylabel("dV (mm)")

    # Resíduo
    ax_resid = axes[2, col_idx]
    ax_resid.plot(res.resid, color=cluster_color, linewidth=1.8)
    ax_resid.set_title(f"Resíduo – Cluster {cluster_id}", fontsize=12)
    ax_resid.set_ylim(resid_min, resid_max)
    if col_idx == 0:
        ax_resid.set_ylabel("dV (mm)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL das Séries Médias por Cluster (dV)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais para dV
agg_pivot_dv = agg.pivot(index='cell_id', columns='date', values='dV').fillna(0)
cell_ids = agg_pivot_dv.index.tolist()
X = agg_pivot_dv.values.astype(float)
n = X.shape[0]


# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11.1. Dendrograma + Mapa de Clusters (dV)
# ==============================
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram

clusters_present = sorted(np.unique(cluster_labels))
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Atualizar cores
palette = sns.color_palette("Set2", len(clusters_present))
cluster_colors = {cl: to_hex(c) for cl, c in zip(clusters_present, palette)}

# --- Função auxiliar para colorir ramos do dendrograma
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Define a cor de cada ramo com base no cluster predominante abaixo do corte."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# --- Dicionário com o cluster de cada célula
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

# --- Figura combinada: Dendrograma + Mapa ---
fig = plt.figure(figsize=(22, 11))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.03)

# --- Dendrograma (esquerda)
ax_dendro = fig.add_subplot(gs[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.3, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels conforme o cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma Hierárquico (DTW – dV)", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig.add_subplot(gs[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=0.5, alpha=0.3)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# Legenda
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} ({(cluster_df["cluster"]==cluster_id).sum()} células)')
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical (dV)\n"
    f"DTW + agrupamento hierárquico (ligação média)",
    fontsize=16
)

plt.tight_layout()
plt.show()


# ==============================
# 12. Figura: mapa + séries com nível (com legenda e caixa técnica)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters, pontos e caixa técnica ---
ax_map = fig.add_subplot(gs[0, :])

# Contornos base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

# Células coloridas conforme o cluster
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

# Pontos centrais (pontos ASC/DESC reais)
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# --- Handles para legenda ---
handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(
        plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}')
    )

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# # Caixa técnica (informações do estudo)
# textstr = '\n'.join((
#     f'Grelha: {grid_size} m x {grid_size} m',
#     f'Clustering: DTW + Hierárquico (ligação média)',
#     'Tipo de deslocamento: dV (vertical)'
# ))
# ax_map.text(0.99, 0.01, textstr, transform=ax_map.transAxes,
#             fontsize=10, verticalalignment='bottom', horizontalalignment='right',
#             bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento vertical.\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série temporal média de cada cluster e a temperatura.",
    fontsize=16
)

ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries temporais médias por cluster + temperatura ---
dV_min = agg['dV'].min()
dV_max = agg['dV'].max()
dV_margin = (dV_max - dV_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]

    # Séries individuais do cluster (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster (colorida)
    cluster_mean_dV = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dV, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário, preta)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Limites dos eixos
    ax.set_ylim(dV_min - dV_margin, dV_max + dV_margin)
    ax2.set_ylim(dV_min - dV_margin, dV_max + dV_margin)

    # Ticks reais da temperatura
    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dV_max - dV_min) * 0.25 + (dV_max - (dV_max - dV_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dV (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    # Legenda combinada
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()

from statsmodels.tsa.seasonal import STL

# ==============================
# 13. Decomposição STL das séries médias por cluster
# ==============================

# Ordenar clusters: cluster 1 vem sempre primeiro
clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered  # cluster 1 na primeira coluna

fig_stl, axes = plt.subplots(3, n_clusters, figsize=(5 * n_clusters, 10), sharex=True)

# Garantir formato correto se houver apenas um cluster
if n_clusters == 1:
    axes = np.array([axes]).T

# Calcular decomposições STL e limites globais
stl_results = {}
trend_min, trend_max = np.inf, -np.inf
season_min, season_max = np.inf, -np.inf
resid_min, resid_max = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dv.loc[cluster_cells]
    cluster_mean_dV = cluster_data.mean(axis=0)

    ts = pd.Series(cluster_mean_dV.values, index=pd.to_datetime(cluster_mean_dV.index)).asfreq('MS').interpolate()
    stl = STL(ts, period=12, robust=True)
    res = stl.fit()
    stl_results[cluster_id] = res

    # Atualizar limites globais para manter mesma escala entre clusters
    trend_min = min(trend_min, res.trend.min())
    trend_max = max(trend_max, res.trend.max())
    season_min = min(season_min, res.seasonal.min())
    season_max = max(season_max, res.seasonal.max())
    resid_min = min(resid_min, res.resid.min())
    resid_max = max(resid_max, res.resid.max())

# Plot de cada cluster (mesma cor por cluster)
for col_idx, cluster_id in enumerate(clusters_ordered):
    res = stl_results[cluster_id]
    cluster_color = cluster_colors[cluster_id]

    # Tendência
    ax_trend = axes[0, col_idx]
    ax_trend.plot(res.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_title(f"Tendência – Cluster {cluster_id}", fontsize=12)
    ax_trend.set_ylim(trend_min, trend_max)
    if col_idx == 0:
        ax_trend.set_ylabel("dV (mm)")

    # Sazonal
    ax_seasonal = axes[1, col_idx]
    ax_seasonal.plot(res.seasonal, color=cluster_color, linewidth=2.0)
    ax_seasonal.set_title(f"Sazonal – Cluster {cluster_id}", fontsize=12)
    ax_seasonal.set_ylim(season_min, season_max)
    if col_idx == 0:
        ax_seasonal.set_ylabel("dV (mm)")

    # Resíduo
    ax_resid = axes[2, col_idx]
    ax_resid.plot(res.resid, color=cluster_color, linewidth=1.8)
    ax_resid.set_title(f"Resíduo – Cluster {cluster_id}", fontsize=12)
    ax_resid.set_ylim(resid_min, resid_max)
    if col_idx == 0:
        ax_resid.set_ylabel("dV (mm)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL das Séries Médias por Cluster (dV)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# DTW+hier, ligação média, dH, STL

In [ ]:
# ==============================
# 0. Bibliotecas
# ==============================
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree
from scipy.signal import savgol_filter
from sklearn.cluster import KMeans
import matplotlib.dates as mdates

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

def filter_area(df):
    return df[(df['northing'] >= norte_min) & (df['northing'] <= norte_max) &
              (df['easting'] >= este_min) & (df['easting'] <= este_max)]

asc = filter_area(asc)
desc = filter_area(desc)
ortho_v = filter_area(ortho_v)
ortho_h = filter_area(ortho_h)

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # ajustar conforme colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Criar grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

# Remover NaNs antes de criar cell_id
asc_interp = asc_interp.dropna(subset=['cell_x','cell_y'])
asc_interp['cell_id'] = asc_interp['cell_x'].astype(int).astype(str) + "_" + asc_interp['cell_y'].astype(int).astype(str)

# Agrupar para ponto central de cada célula
points = asc_interp.groupby('cell_id').agg({'easting':'mean','northing':'mean'}).reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['easting'], points['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

agg = asc_interp.groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. Criar GeoDataFrame da grelha
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)

# ==============================
# 10. Carregar temperatura
# ==============================
df_temp = pd.read_excel("data/alqueva_temp.xlsx")
df_temp['data'] = pd.to_datetime(df_temp['data'])
window = 365
df_temp['med_smooth'] = savgol_filter(df_temp['med'], window_length=window, polyorder=2)

# ==============================
# 11. Clustering DTW + Hierárquico
# ==============================
from dtaidistance import dtw
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
import seaborn as sns
from matplotlib.colors import to_hex

# Pivot das séries temporais para dH
agg_pivot_dh = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)
cell_ids = agg_pivot_dh.index.tolist()
X = agg_pivot_dh.values.astype(float)
n = X.shape[0]


# Normalizar séries
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Calcular matriz de distâncias DTW (usar janela para acelerar)
dist_matrix = np.zeros((n, n))
window = int(0.1 * X_scaled.shape[1])  # 10% da série

for i in range(n):
    xi = X_scaled[i, :]
    for j in range(i + 1, n):
        xj = X_scaled[j, :]
        dist_matrix[i, j] = dtw.distance(xi, xj, window=window)
        dist_matrix[j, i] = dist_matrix[i, j]

# Converter para formato condensado
condensed = squareform(dist_matrix)

# Clustering hierárquico (método 'average' recomendado para DTW)
Z = linkage(condensed, method='average')

# Corte automático pelo "cotovelo"
distances = Z[:, 2]
diffs = np.diff(distances)
max_jump_idx = np.argmax(diffs)
cut_distance = (distances[max_jump_idx] + distances[max_jump_idx - 1]) / 2

# Labels de clusters
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
num_clusters = len(np.unique(cluster_labels))
print(f"Corte ajustado em {cut_distance:.3f}, clusters encontrados: {num_clusters}")

# Criar dataframe de clusters
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Mapear cores (mesmas cores do teu gráfico)
palette = sns.color_palette("Set2", num_clusters)
cluster_colors = {cl: to_hex(c) for cl, c in zip(np.unique(cluster_labels), palette)}

grid_sel = grid.merge(cluster_df, on='cell_id', how='left')

# ==============================
# 11.1. Dendrograma + Mapa de Clusters (dH)
# ==============================
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram

# Recalcular clusters se ainda não estiverem definidos
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
clusters_present = sorted(np.unique(cluster_labels))
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Atualizar cores
palette = sns.color_palette("Set2", len(clusters_present))
cluster_colors = {cl: to_hex(c) for cl, c in zip(clusters_present, palette)}

# Função auxiliar para colorir ramos
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme o cluster dominante."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# --- Figura: Dendrograma + Mapa ---
fig = plt.figure(figsize=(22, 11))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.03)

# --- Dendrograma (esquerda)
ax_dendro = fig.add_subplot(gs[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.3, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels conforme cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma Hierárquico (DTW)", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig.add_subplot(gs[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=0.5, alpha=0.3)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# Legenda
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} ({(cluster_df["cluster"]==cluster_id).sum()} células)')
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento horizontal (dH)\n"
    f"DTW + agrupamento hierárquico (ligação média)",
    fontsize=16
)

plt.tight_layout()
plt.show()


# ==============================
# 12. Figura: mapa + séries com nível (para dH)
# ==============================
clusters_present = sorted(cluster_df['cluster'].unique())
n_clusters = len(clusters_present)

# Pivot dH
agg_pivot_dh = agg.pivot(index='cell_id', columns='date', values='dH').fillna(0)

fig = plt.figure(figsize=(20, 18))
gs = fig.add_gridspec(2, n_clusters, height_ratios=[2, 1.2])

# --- Linha 1: Mapa com clusters ---
ax_map = fig.add_subplot(gs[0, :])

grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=1, alpha=0.2)

for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map, color=cluster_colors[int(row['cluster'])], alpha=0.4
        )

gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

handles = [
    plt.Line2D([0], [0], marker='o', color='white', markeredgecolor='black',
               markersize=10, linestyle='None', label='Pontos ASC/DESC')
]
for cl, color in cluster_colors.items():
    handles.append(plt.Line2D([0], [0], color=color, linewidth=8, label=f'Cluster {cl}'))

ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

ax_map.set_title(
    f"Clusters de séries temporais de deslocamento horizontal (dH).\n"
    f"DTW + agrupamento hierárquico, ligação média.\n"
    f"Grelha {grid_size} m × {grid_size} m.\n"
    f"Correlação entre a série média de cada cluster e a temperatura.",
    fontsize=16
)
ax_map.set_axis_off()
ax_map.legend(handles=handles, loc='upper left', fontsize=9, framealpha=0.9)

# --- Linha 2: Séries médias por cluster + temperatura ---
dH_min = agg['dH'].min()
dH_max = agg['dH'].max()
dH_margin = (dH_max - dH_min) * 0.1

temp_min = df_temp['med_smooth'].min()
temp_max = df_temp['med_smooth'].max()

for idx, cluster_id in enumerate(clusters_present):
    ax = fig.add_subplot(gs[1, idx])
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dh.loc[cluster_cells]

    # Séries individuais (cinza)
    for cid in cluster_cells:
        ax.plot(cluster_data.columns, cluster_data.loc[cid], color='lightgray', alpha=0.7)

    # Média do cluster
    cluster_mean_dH = cluster_data.mean(axis=0)
    cluster_color = cluster_colors[cluster_id]
    ax.plot(cluster_data.columns, cluster_mean_dH, color=cluster_color, linewidth=2.5,
            label=f'Média Cluster {cluster_id}')

    # Temperatura suavizada (eixo secundário)
    ax2 = ax.twinx()
    temp_norm = (df_temp['med_smooth'] - temp_min) / (temp_max - temp_min)
    temp_visual = temp_norm * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)
    ax2.plot(df_temp['data'], temp_visual, color='black', linewidth=2.2, alpha=0.85,
             label='Temperatura média (°C)')

    # Eixos e limites
    ax.set_ylim(dH_min - dH_margin, dH_max + dH_margin)
    ax2.set_ylim(dH_min - dH_margin, dH_max + dH_margin)

    temp_ticks_real = np.linspace(temp_min, temp_max, 6)
    temp_ticks_visual = (temp_ticks_real - temp_min) / (temp_max - temp_min)
    temp_ticks_visual = temp_ticks_visual * (dH_max - dH_min) * 0.25 + (dH_max - (dH_max - dH_min) * 0.3)
    ax2.set_yticks(temp_ticks_visual)
    ax2.set_yticklabels([f"{t:.1f}" for t in temp_ticks_real])
    ax2.set_ylabel("Temperatura (°C)")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.set_ylabel("dH (mm)")
    ax.set_title(f'Cluster {cluster_id} - {len(cluster_cells)} células', fontsize=12)
    ax.grid(False)
    ax2.grid(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


# ==============================
# 13. Decomposição STL das séries médias por cluster (dH)
# ==============================
from statsmodels.tsa.seasonal import STL

clusters_ordered = sorted(clusters_present)
if 1 in clusters_ordered:
    clusters_ordered.remove(1)
    clusters_ordered = [1] + clusters_ordered

fig_stl, axes = plt.subplots(3, n_clusters, figsize=(5 * n_clusters, 10), sharex=True)
if n_clusters == 1:
    axes = np.array([axes]).T

stl_results = {}
trend_min, trend_max = np.inf, -np.inf
season_min, season_max = np.inf, -np.inf
resid_min, resid_max = np.inf, -np.inf

for cluster_id in clusters_ordered:
    cluster_cells = cluster_df[cluster_df['cluster'] == cluster_id]['cell_id'].tolist()
    cluster_data = agg_pivot_dh.loc[cluster_cells]
    cluster_mean_dH = cluster_data.mean(axis=0)

    ts = pd.Series(cluster_mean_dH.values, index=pd.to_datetime(cluster_mean_dH.index)).asfreq('MS').interpolate()
    stl = STL(ts, period=12, robust=True)
    res = stl.fit()
    stl_results[cluster_id] = res

    trend_min = min(trend_min, res.trend.min())
    trend_max = max(trend_max, res.trend.max())
    season_min = min(season_min, res.seasonal.min())
    season_max = max(season_max, res.seasonal.max())
    resid_min = min(resid_min, res.resid.min())
    resid_max = max(resid_max, res.resid.max())

for col_idx, cluster_id in enumerate(clusters_ordered):
    res = stl_results[cluster_id]
    cluster_color = cluster_colors[cluster_id]

    ax_trend = axes[0, col_idx]
    ax_trend.plot(res.trend, color=cluster_color, linewidth=2.5)
    ax_trend.set_title(f"Tendência – Cluster {cluster_id}", fontsize=12)
    ax_trend.set_ylim(trend_min, trend_max)
    if col_idx == 0:
        ax_trend.set_ylabel("dH (mm)")

    ax_seasonal = axes[1, col_idx]
    ax_seasonal.plot(res.seasonal, color=cluster_color, linewidth=2.0)
    ax_seasonal.set_title(f"Sazonal – Cluster {cluster_id}", fontsize=12)
    ax_seasonal.set_ylim(season_min, season_max)
    if col_idx == 0:
        ax_seasonal.set_ylabel("dH (mm)")

    ax_resid = axes[2, col_idx]
    ax_resid.plot(res.resid, color=cluster_color, linewidth=1.8)
    ax_resid.set_title(f"Resíduo – Cluster {cluster_id}", fontsize=12)
    ax_resid.set_ylim(resid_min, resid_max)
    if col_idx == 0:
        ax_resid.set_ylabel("dH (mm)")

for ax in axes.flatten():
    ax.grid(False)

plt.suptitle("Decomposição STL das Séries Médias por Cluster (dH)", fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()



In [ ]:
# ==============================
# 11.1. Dendrograma + Mapa de Clusters (dH)
# ==============================
import matplotlib.gridspec as gridspec
from scipy.cluster.hierarchy import dendrogram

# Recalcular clusters se ainda não estiverem definidos
cluster_labels = fcluster(Z, t=cut_distance, criterion='distance')
clusters_present = sorted(np.unique(cluster_labels))
cluster_df = pd.DataFrame({'cell_id': cell_ids, 'cluster': cluster_labels})

# Atualizar cores
palette = sns.color_palette("Set2", len(clusters_present))
cluster_colors = {cl: to_hex(c) for cl, c in zip(clusters_present, palette)}

# Função auxiliar para colorir ramos
def branch_color(y_coords, x_coords, ivl, leaf_clusters, cut_distance):
    """Determina a cor de cada ramo do dendrograma conforme o cluster dominante."""
    if max(y_coords) > cut_distance:
        return "#A0A0A0"  # acima do corte
    leaf_indices = [int(round((i - 5) / 10)) for i in x_coords if 0 <= (i - 5) / 10 < len(ivl)]
    cluster_ids = [leaf_clusters[ivl[i]] for i in leaf_indices if i in range(len(ivl))]
    if cluster_ids:
        cluster_id = max(set(cluster_ids), key=cluster_ids.count)
        return cluster_colors.get(cluster_id, "#A0A0A0")
    return "#A0A0A0"

# --- Figura: Dendrograma + Mapa ---
fig = plt.figure(figsize=(22, 11))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 3], wspace=0.03)

# --- Dendrograma (esquerda)
ax_dendro = fig.add_subplot(gs[0, 0])
dendro = dendrogram(Z, labels=cell_ids, orientation='right', no_plot=True)
leaf_clusters = {cell_id: cluster_df.set_index('cell_id').loc[cell_id, 'cluster'] for cell_id in cell_ids}

for xs, ys in zip(dendro['icoord'], dendro['dcoord']):
    color = branch_color(ys, xs, dendro['ivl'], leaf_clusters, cut_distance)
    ax_dendro.plot(ys, xs, color=color, linewidth=2.3, solid_capstyle='round', zorder=3)

ax_dendro.axvline(x=cut_distance, color='red', linestyle='--', linewidth=2, label='Corte DTW')
ax_dendro.set_xlabel("Distância DTW")
ax_dendro.set_ylabel("cell_id")

# Colorir labels conforme cluster
ax_dendro.set_yticks(range(5, 10 * len(dendro['ivl']) + 5, 10))
ax_dendro.set_yticklabels(dendro['ivl'], fontsize=8)
for ticklabel in ax_dendro.get_yticklabels():
    label = ticklabel.get_text()
    cluster_id = leaf_clusters.get(label, 0)
    ticklabel.set_color(cluster_colors.get(cluster_id, "#333333"))

ax_dendro.set_title("Dendrograma Hierárquico (DTW)", fontsize=14)
ax_dendro.legend(fontsize=10, loc='lower right')

# --- Mapa (direita)
ax_map = fig.add_subplot(gs[0, 1])
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5)
grid_sel.boundary.plot(ax=ax_map, color='black', linewidth=0.5, alpha=0.3)
for _, row in grid_sel.iterrows():
    if pd.notna(row['cluster']):
        gpd.GeoSeries([row['geometry']], crs=grid_sel.crs).plot(
            ax=ax_map,
            color=cluster_colors[int(row['cluster'])],
            alpha=0.4
        )

# Pontos centrais
gdf_points.plot(ax=ax_map, color='white', edgecolor='black', markersize=40)

# Basemap
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)

# Legenda
for cluster_id in clusters_present:
    color = cluster_colors[cluster_id]
    ax_map.scatter([], [], color=color, alpha=0.6,
                   label=f'Cluster {cluster_id} ({(cluster_df["cluster"]==cluster_id).sum()} células)')
ax_map.scatter([], [], marker='o', color='white', edgecolor='black', s=120,
               label='Pontos ASC/DESC')

ax_map.legend(fontsize=10, loc='upper left')
ax_map.set_axis_off()
ax_map.set_title(
    f"Clusters de séries temporais de deslocamento horizontal (dH)\n"
    f"DTW + agrupamento hierárquico (ligação média)",
    fontsize=16
)

plt.tight_layout()
plt.show()
